# 🛰️ Satellite Image-Based Flood Risk Mapping
## Vision Transformers for Flood Detection — Step-by-Step Notebook

This notebook walks through the complete pipeline:
1. Data generation & preprocessing
2. Model training (ViT & CNN)
3. Evaluation & comparison
4. Prediction & visualization
5. Attention map analysis

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, '..')  # Add project root

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

## 1. Generate Demo Data

In [ ]:
from src.utils.generate_demo_data import generate_demo_data
from src.preprocessing.dataset import prepare_dataset

# Generate 150 images per class
generate_demo_data(n_per_class=150)

# Split into train/val/test
prepare_dataset('data/raw', 'data/processed')
print('\nDataset ready!')

## 2. Visualize Sample Images

In [ ]:
from src.preprocessing.dataset import FloodDataset, CLASS_NAMES
from src.preprocessing.augmentation import get_val_transforms, denormalize

dataset = FloodDataset('data/processed/train', transform=get_val_transforms())
print(f'Train samples: {len(dataset)}')

# Show one sample per class
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
shown = set()
for img_tensor, label in dataset:
    if label not in shown:
        img = denormalize(img_tensor)
        axes[label].imshow(img)
        axes[label].set_title(CLASS_NAMES[label], fontsize=11, fontweight='bold')
        axes[label].axis('off')
        shown.add(label)
    if len(shown) == 5:
        break

plt.suptitle('Sample Images by Flood Risk Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/plots/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Build DataLoaders

In [ ]:
from src.preprocessing.dataset import get_dataloaders

loaders = get_dataloaders('data/processed', batch_size=16)

# Inspect a batch
images, labels = next(iter(loaders['train']))
print(f'Batch shape: {images.shape}')  # (16, 3, 224, 224)
print(f'Labels: {labels[:8].tolist()}')
print(f'Classes: {[CLASS_NAMES[l] for l in labels[:8].tolist()]}')

## 4. Build & Inspect Models

In [ ]:
from src.models.model_factory import get_model, get_model_summary

# Vision Transformer
vit_model = get_model('vit', device=device)
vit_summary = get_model_summary(vit_model)
print('\nViT Summary:')
for k, v in vit_summary.items():
    print(f'  {k}: {v:,}' if isinstance(v, int) else f'  {k}: {v:.1f}%')

# CNN
cnn_model = get_model('cnn', device=device)
cnn_summary = get_model_summary(cnn_model)
print('\nCNN Summary:')
for k, v in cnn_summary.items():
    print(f'  {k}: {v:,}' if isinstance(v, int) else f'  {k}: {v:.1f}%')

## 5. Train Vision Transformer

In [ ]:
from src.utils.trainer import Trainer

class_weights = loaders['train'].dataset.get_class_weights()
print(f'Class weights: {class_weights.numpy().round(3)}')

vit_trainer = Trainer(
    model=vit_model,
    train_loader=loaders['train'],
    val_loader=loaders['val'],
    model_name='vit',
    class_weights=class_weights,
    device=device,
)

vit_history = vit_trainer.train(epochs=10)  # Use 30 for full training

## 6. Train CNN Baseline

In [ ]:
cnn_trainer = Trainer(
    model=cnn_model,
    train_loader=loaders['train'],
    val_loader=loaders['val'],
    model_name='cnn',
    class_weights=class_weights,
    device=device,
)

cnn_history = cnn_trainer.train(epochs=10)

## 7. Plot Training Curves

In [ ]:
from src.visualization.risk_map import plot_training_history, plot_model_comparison

plot_training_history(vit_history, 'ViT-B/16', 'outputs/plots/vit_history.png')
plot_training_history(cnn_history, 'EfficientNet-B3', 'outputs/plots/cnn_history.png')
plot_model_comparison(vit_history, cnn_history, 'outputs/plots/comparison.png')
plt.show()

## 8. Evaluate Models

In [ ]:
from src.utils.evaluator import evaluate_model, compare_models, print_evaluation_report, print_comparison_report
from src.visualization.risk_map import plot_confusion_matrix

# Load best checkpoints
vit_trainer.load_best_checkpoint()
cnn_trainer.load_best_checkpoint()

vit_results = evaluate_model(vit_model, loaders['test'], device)
cnn_results = evaluate_model(cnn_model, loaders['test'], device)

print_evaluation_report(vit_results, 'ViT-B/16')
print_evaluation_report(cnn_results, 'EfficientNet-B3')

comparison = compare_models(vit_results, cnn_results)
print_comparison_report(comparison)

## 9. Confusion Matrices

In [ ]:
fig_vit = plot_confusion_matrix(vit_results['confusion_matrix'], model_name='ViT-B/16')
fig_cnn = plot_confusion_matrix(cnn_results['confusion_matrix'], model_name='EfficientNet-B3')
plt.show()

## 10. Single Image Prediction & Visualization

In [ ]:
from PIL import Image
import numpy as np
from src.utils.predictor import FloodPredictor
from src.preprocessing.augmentation import get_val_transforms
from src.visualization.heatmap import extract_vit_attention, plot_probability_bars, apply_heatmap_overlay
from src.visualization.risk_map import plot_risk_map

# Find a test image
test_imgs = list(Path('data/processed/test').rglob('*.jpg'))
test_img_path = str(test_imgs[0]) if test_imgs else None

if test_img_path:
    # Load image
    pil_img = Image.open(test_img_path).convert('RGB').resize((224, 224))
    image_np = np.array(pil_img)
    transform = get_val_transforms()
    tensor = transform(pil_img).unsqueeze(0)

    # Predict with ViT
    predictor_vit = FloodPredictor(vit_model, 'vit', device)
    result = predictor_vit.predict(pil_img)

    print(f'Predicted: {result.predicted_class} ({result.confidence:.1%})')
    print(f'Risk Level: {result.risk_level}')
    print(f'Alert: {result.alert}')

    # Probability bars
    fig = plot_probability_bars(result.class_probabilities, title=f'Prediction: {result.predicted_class}')
    plt.show()

    # ViT Attention Map
    attn_map = extract_vit_attention(vit_model, tensor.to(device))
    overlay = apply_heatmap_overlay(image_np, attn_map)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(image_np)
    axes[0].set_title('Original')
    axes[0].axis('off')
    axes[1].imshow(overlay)
    axes[1].set_title('ViT Attention Rollout')
    axes[1].axis('off')
    plt.suptitle(f'Prediction: {result.predicted_class}', fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Risk zone map
    plot_risk_map(image_np, attn_map, title=f'Flood Risk Map — {result.predicted_class}')
    plt.show()
else:
    print('No test images found. Run data preparation first.')

## 11. Launch Dashboard

Run this in terminal:
```bash
streamlit run dashboard/app.py
```
Then open: http://localhost:8501